In [1]:
import itertools
import os
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.parameters import Flux_IEEEMachine2, IEEEMachine2
from current_setpoints.optimization import ModelAnalytical
from current_setpoints.utils import (
    NeuralTorquePredictor,
    evaluate_model,
    load_aggregated_csv_data,
    prepare_fold_dataloaders,
    train_model,
)

In [2]:
AGGREGATED_FILE_PATH = "../data/aggregated_file_means.csv"
TARGET_VARIABLE = "torq"
COLUMN_MAP = {
    "omega": "omega",
    "id1": "id1",
    "iq1": "iq1",
    "id3": "id3",
    "iq3": "iq3",
    "torq": "torq",
}
INPUT_SIZE = 5

K_SPLITS = 5
TEST_SIZE = 0.15
CV_EPOCHS = 200
CV_PATIENCE = 10

HIDDEN_SIZES = [12]
LEARNING_RATES = [1e-3, 3e-3, 5e-3, 6e-3,  8e-3, 9e-3, 1e-2, 1.5e-2, 2e-2, 5e-2, 1e-1]
REG_LAMBDAS = [5e-7, 1e-6, 5e-6,1e-5, 5e-5, 1e-4, 2e-4]

FINAL_EPOCHS = 800
FINAL_BATCH_SIZE = 64
FINAL_PATIENCE = 15
MIN_DELTA = 1e-5
MODEL_SAVE_PATH = "../weights/NTM_Weights_fin.pth"
SCALER_SAVE_PATH = "../weights/NTM_Scaler_fin.npy"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {DEVICE}")

Using compute device: cuda


In [3]:
# Note: the new neural model is purely a residual on top of the analytical
# torque (the analytical part is added inside ``ModelNeural.calculate_torque``
# at inference time). To stay consistent with that, the network is trained
# against the residual y = measured - analytical, NOT against raw measured
# torque. If you trained against measured torque directly, inference would
# return analytical + network(x) ~= 2*analytical + true_residual.
flux = Flux_IEEEMachine2()
machine = IEEEMachine2()
# Operating limits are not set in the IEEEMachine2 constructor; they must be
# declared explicitly before any consumer (here ModelAnalytical) reads them.
machine.set_max_pars(curr_max=30.0, volt_max=13.0, omega_max=1800)
analytical_model = ModelAnalytical(machine=machine, flux=flux)

data = load_aggregated_csv_data(AGGREGATED_FILE_PATH, COLUMN_MAP)
X_features = ["omega", "id1", "iq1", "id3", "iq3"]

X = data[X_features].values.astype(np.float32)
y_measured = data[[TARGET_VARIABLE]].values.astype(np.float32)

print("Pre-calculating analytical torque baseline for residual training...")
y_analytical = np.array(
    [
        analytical_model.calculate_torque(
            float(row[0]), row[1:].astype(np.float64)
        )
        for row in X
    ],
    dtype=np.float32,
).reshape(-1, 1)
y = y_measured - y_analytical

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42
)

print(f"Training/Validation Base Set Size: {X_train_val.shape[0]}")
print(f"Holdout Test Set Size: {X_test.shape[0]}")
print(
    f"Residual target stats - mean: {y.mean():.4f}, std: {y.std():.4f}, "
    f"min: {y.min():.4f}, max: {y.max():.4f}"
)

Loaded 174 valid data points from CSV.
Pre-calculating analytical torque baseline for residual training...
Training/Validation Base Set Size: 147
Holdout Test Set Size: 27
Residual target stats - mean: -0.0649, std: 0.1991, min: -0.5902, max: 0.4541


In [4]:
kf = KFold(n_splits=K_SPLITS, shuffle=True, random_state=42)
hyperparameters = list(itertools.product(HIDDEN_SIZES, LEARNING_RATES, REG_LAMBDAS))
results_list = []

print(f"Total Combinations to Test: {len(hyperparameters)}")

for iteration, (h_size, lr, reg) in enumerate(hyperparameters):
    print(
        f"\n*** Combination {iteration + 1}/{len(hyperparameters)} | "
        f"H_Size: {h_size}, LR: {lr:.1e}, Reg: {reg:.1e} ***"
    )
    cv_performance = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_val)):
        train_loader, val_loader, scaler_X = prepare_fold_dataloaders(
            X_train_val[train_idx],
            X_train_val[val_idx],
            y_train_val[train_idx],
            y_train_val[val_idx],
        )

        model = NeuralTorquePredictor(
            input_size=INPUT_SIZE,
            hidden_size=h_size,
            scaler_X=scaler_X,
            device=DEVICE,
        ).to(DEVICE)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=reg)
        criterion = nn.MSELoss()

        best_loss, _ = train_model(
            model,
            train_loader,
            val_loader,
            criterion,
            optimizer,
            CV_EPOCHS,
            CV_PATIENCE,
            DEVICE,
        )
        cv_performance.append(np.sqrt(best_loss))

    mean_rmse, std_rmse = np.mean(cv_performance), np.std(cv_performance)
    results_list.append(
        {"H_Size": h_size, "LR": lr, "Reg_Lambda": reg, "Mean_CV_RMSE": mean_rmse}
    )
    print(f"  --> Mean CV RMSE: {mean_rmse:.4f} (+/- {std_rmse:.4f})")

results_df = pd.DataFrame(results_list).sort_values(by="Mean_CV_RMSE")
best_params = results_df.iloc[0]
print("\n=======================================================================")
print(f"BEST MEAN CV RMSE Found: {best_params['Mean_CV_RMSE']:.4f} Nm")
print(
    f"CV-suggested params -> H_Size: {int(best_params['H_Size'])}, "
    f"LR: {best_params['LR']:.1e}, Reg: {best_params['Reg_Lambda']:.1e}"
)
print("=======================================================================")
results_df.head()

Total Combinations to Test: 77

*** Combination 1/77 | H_Size: 12, LR: 1.0e-03, Reg: 5.0e-07 ***

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 86!

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 139!

Starting Training with Early Stopping (Patience=10)...

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 119!

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 180!
  --> Mean CV RMSE: 0.0487 (+/- 0.0156)

*** Combination 2/77 | H_Size: 12, LR: 1.0e-03, Reg: 1.0e-06 ***

Starting Training with Early Stopping (Patience=10)...

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 186!

Starting Training with Early Stopping (Patience=10)...

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 129!

Starting Training with Early Stopping (Patience=10)..

,H_Size,LR,Reg_Lambda,Mean_CV_RMSE
57,12,0.020,1.000000e-06,0.036509
52,12,0.015,1.000000e-05,0.037087
46,12,0.010,5.000000e-05,0.037258
56,12,0.020,5.000000e-07,0.037393
62,12,0.020,2.000000e-04,0.037651


In [17]:
# Final-training hyperparameters used for the published model.
# These are intentionally fixed (not auto-pulled from the CV result above)
# so re-running CV on a slightly different machine cannot change which
# weights ship with the article. To re-tune, replace with values from
# `best_params` and rerun.

OPTIMAL_HIDDEN_SIZE = 12
OPTIMAL_LEARNING_RATE = 2e-2
OPTIMAL_REG_LAMBDA = 1e-6

print(
    f"Final-training hyperparameters: H_Size={OPTIMAL_HIDDEN_SIZE}, "
    f"LR={OPTIMAL_LEARNING_RATE:.1e}, Reg={OPTIMAL_REG_LAMBDA:.1e}"
)

Final-training hyperparameters: H_Size=12, LR=2.0e-02, Reg=1.0e-06


In [18]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=42
)

scaler_X = StandardScaler()
X_train_norm = scaler_X.fit_transform(X_train)
X_val_norm = scaler_X.transform(X_val)
X_test_norm = scaler_X.transform(X_test)

train_dataset = TensorDataset(
    torch.from_numpy(X_train_norm).float(),
    torch.from_numpy(y_train).float(),
)
val_dataset = TensorDataset(
    torch.from_numpy(X_val_norm).float(),
    torch.from_numpy(y_val).float(),
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test_norm).float(),
    torch.from_numpy(y_test).float(),
)

train_loader = DataLoader(train_dataset, batch_size=FINAL_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=FINAL_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=FINAL_BATCH_SIZE, shuffle=False)

In [19]:
TARGET_RMSE = 0.040
attempt = 0
final_test_rmse = float("inf")

final_model = None
epochs_run = 0
best_val_loss = float("inf")

while final_test_rmse >= TARGET_RMSE:
    attempt += 1
    print(f"\n{'='*70}")
    print(f"  ATTEMPT {attempt} — Target RMSE < {TARGET_RMSE}")
    print(f"{'='*70}")

    final_model = NeuralTorquePredictor(
        input_size=INPUT_SIZE,
        hidden_size=OPTIMAL_HIDDEN_SIZE,
        scaler_X=scaler_X,
        device=DEVICE,
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        final_model.parameters(),
        lr=OPTIMAL_LEARNING_RATE,
        weight_decay=OPTIMAL_REG_LAMBDA,
    )

    print(
        f"Starting training with H_Size: {OPTIMAL_HIDDEN_SIZE}, "
        f"LR: {OPTIMAL_LEARNING_RATE:.1e}, Reg: {OPTIMAL_REG_LAMBDA:.1e}"
    )

    best_val_loss, epochs_run = train_model(
        model=final_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=FINAL_EPOCHS,
        patience=FINAL_PATIENCE,
        device=DEVICE,
        min_delta=MIN_DELTA,
        verbose=True,
    )

    final_test_rmse = evaluate_model(final_model, test_loader, criterion, DEVICE)

    print(f"\nAttempt {attempt} — Test RMSE: {final_test_rmse:.4f} Nm  (target: < {TARGET_RMSE})")

    if final_test_rmse >= TARGET_RMSE:
        print("Target not met, retraining from scratch...")

assert final_model is not None, "Model failed to initialize."

torch.save(final_model.state_dict(), MODEL_SAVE_PATH)
scaler_data = {"mean": scaler_X.mean_, "scale": scaler_X.scale_}
np.save(SCALER_SAVE_PATH, np.array(scaler_data, dtype=object), allow_pickle=True)

print("\n=======================================================================")
print("            FINAL NTM MODEL PERFORMANCE REPORT")
print("=======================================================================")
print(f"Succeeded on attempt {attempt}.")
print(f"Training completed in {epochs_run} epochs. Best validation loss: {best_val_loss:.6f}")
print(f"FINAL TEST SET RMSE (generalization metric): {final_test_rmse:.4f} Nm")
print("=======================================================================")


  ATTEMPT 1 — Target RMSE < 0.04
Starting training with H_Size: 12, LR: 2.0e-02, Reg: 1.0e-06

Starting Training with Early Stopping (Patience=15)...

Early stopping triggered at epoch 96!

Attempt 1 — Test RMSE: 0.0433 Nm  (target: < 0.04)
Target not met, retraining from scratch...

  ATTEMPT 2 — Target RMSE < 0.04
Starting training with H_Size: 12, LR: 2.0e-02, Reg: 1.0e-06

Starting Training with Early Stopping (Patience=15)...

Early stopping triggered at epoch 126!

Attempt 2 — Test RMSE: 0.0463 Nm  (target: < 0.04)
Target not met, retraining from scratch...

  ATTEMPT 3 — Target RMSE < 0.04
Starting training with H_Size: 12, LR: 2.0e-02, Reg: 1.0e-06

Starting Training with Early Stopping (Patience=15)...

Early stopping triggered at epoch 37!

Attempt 3 — Test RMSE: 0.0422 Nm  (target: < 0.04)
Target not met, retraining from scratch...

  ATTEMPT 4 — Target RMSE < 0.04
Starting training with H_Size: 12, LR: 2.0e-02, Reg: 1.0e-06

Starting Training with Early Stopping (Patience=